In [0]:
%sql

MERGE INTO fraud_detection.gold.fraud_cases AS target

USING (
    SELECT
        CONCAT('CASE-', transaction_id) AS case_id,
        transaction_id,
        customer_id,
        card_id,
        merchant_id,
        amount,
        risk_score,
        risk_level,
        recommended_action,
        'open' AS case_status,
        CAST(NULL AS STRING) AS assigned_to,
        CAST(NULL AS STRING) AS investigator_notes,
        'pending' AS customer_response,
        alert_created_timestamp AS created_timestamp,
        CURRENT_TIMESTAMP() AS updated_timestamp
    FROM fraud_detection.gold.realtime_fraud_alerts
) AS source

ON target.transaction_id = source.transaction_id

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql

MERGE INTO fraud_detection.gold.fraud_disputes AS target

USING (
    SELECT
        CONCAT('DISPUTE-', transaction_id) AS dispute_id,
        case_id,
        transaction_id,
        customer_id,
        card_id,
        amount AS disputed_amount,
        'customer_not_authorized' AS dispute_reason,
        'opened' AS dispute_status,
        'pending_review' AS provisional_credit_status,
        CURRENT_TIMESTAMP() AS created_timestamp,
        CURRENT_TIMESTAMP() AS updated_timestamp
    FROM fraud_detection.gold.fraud_cases
    WHERE case_status = 'confirmed_fraud'
      AND customer_response = 'not_authorized'
) AS source

ON target.transaction_id = source.transaction_id

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql

MERGE INTO fraud_detection.gold.fraud_actions AS target

USING (
    SELECT
        CONCAT('BLOCK-', transaction_id) AS action_id,
        dispute_id,
        customer_id,
        card_id,
        transaction_id,
        'block_card' AS action_type,
        CAST(NULL AS DOUBLE) AS action_amount,
        'pending' AS action_status,
        CURRENT_TIMESTAMP() AS created_timestamp,
        CAST(NULL AS TIMESTAMP) AS completed_timestamp
    FROM fraud_detection.gold.fraud_disputes
    WHERE dispute_status = 'opened'
) AS source

ON target.action_id = source.action_id

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql

MERGE INTO fraud_detection.gold.fraud_actions AS target

USING (
    SELECT
        CONCAT('CREDIT-', transaction_id) AS action_id,
        dispute_id,
        customer_id,
        card_id,
        transaction_id,
        'issue_provisional_credit' AS action_type,
        disputed_amount AS action_amount,
        'pending' AS action_status,
        CURRENT_TIMESTAMP() AS created_timestamp,
        CAST(NULL AS TIMESTAMP) AS completed_timestamp
    FROM fraud_detection.gold.fraud_disputes
    WHERE dispute_status = 'opened'
) AS source

ON target.action_id = source.action_id

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql

MERGE INTO fraud_detection.gold.fraud_actions AS target

USING (
    SELECT
        CONCAT('CREDIT-', transaction_id) AS action_id,
        dispute_id,
        customer_id,
        card_id,
        transaction_id,
        'issue_provisional_credit' AS action_type,
        disputed_amount AS action_amount,
        'pending' AS action_status,
        CURRENT_TIMESTAMP() AS created_timestamp,
        CAST(NULL AS TIMESTAMP) AS completed_timestamp
    FROM fraud_detection.gold.fraud_disputes
    WHERE dispute_status = 'opened'
) AS source

ON target.action_id = source.action_id

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM fraud_detection.gold.fraud_cases)
        AS total_cases,

    (SELECT COUNT(*)
     FROM fraud_detection.gold.fraud_disputes)
        AS total_disputes,

    (SELECT COUNT(*)
     FROM fraud_detection.gold.fraud_actions)
        AS total_actions;